# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook guides you through the loading and exploration of the FAIR^2 dataset using the `mlcroissant` library, referencing all schema elements by their `@id` fields. You'll see how to discover record sets, fields, and columns, inspect example records, and perform data processing--all in line with FAIR data practices.

### Dataset Source
The dataset uses the Croissant schema hosted at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load dataset metadata and data using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview

Review available record sets, fields, and their `@id`s. This helps you understand which tables (record sets) are present and how to access them using their `@id`s.

In [ ]:
# List all record sets with their @id and name (if available)
record_sets = list(dataset.recordsets())
if not record_sets:
    print("No record sets were found in the Croissant schema.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        if 'name' in rs:
            print(f"  Name       : {rs['name']}")
        if 'description' in rs:
            print(f"  Description: {rs['description']}")
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        if fields:
            print("  Fields:")
            for field in fields:
                print(f"    - @id: {field['@id']}, name: {field.get('name', '<none>')}")
        print('-'*50)

Below we inspect an example record set by its `@id`. If no record set is found, adapt this block once IDs become available in the schema.

In [ ]:
# For demonstration, if there's at least one record set, show the first few records by @id.
if record_sets:
    # Use the @id of the first RecordSet, as an example
    first_record_set_id = record_sets[0]['@id']
    print(f"Sample records from RecordSet @id: {first_record_set_id}")
    for i, record in enumerate(dataset.records(record_set=first_record_set_id)):
        print(json.dumps(record, indent=2))
        if i == 2:
            break
else:
    print("No record sets available for sample records display.")

## 3. Data Extraction

Load data from one or more record sets into Pandas DataFrames. Refer to each by its exact `@id`.

In [ ]:
# Prepare to extract all available RecordSets by their @id
dataframes = {}
record_set_ids = [rs['@id'] for rs in record_sets]

for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded DataFrame for RecordSet @id: {rs_id} with shape {df.shape}")
    except Exception as e:
        print(f"Could not load records for RecordSet @id: {rs_id} -- {str(e)}")

# Display columns and head for each DataFrame
for rs_id, df in dataframes.items():
    print(f"\nColumns for {rs_id}: {df.columns.tolist()}")
    display(df.head())

## 4. Exploratory Data Analysis (EDA)

Let's process a numeric field: filtering, normalization, and simple grouping. All operations below reference fields by their `@id` (as in the schema).

In [ ]:
# Example: EDA on one record set if available
if dataframes:
    # Use the first record set as demo
    example_record_set_id = list(dataframes.keys())[0]
    df = dataframes[example_record_set_id]
    # Display numeric columns (by @id)
    numeric_columns = []
    for col in df.columns:
        # Try to infer numeric columns (int/float types after dropna)
        try:
            if pd.api.types.is_numeric_dtype(df[col].dropna()):
                numeric_columns.append(col)
        except:
            pass
    if not numeric_columns:
        print("No numeric fields were found in the first record set.")
    else:
        numeric_field_id = numeric_columns[0]
        print(f"Using numeric field (by @id): {numeric_field_id}")
        # Filter by a threshold (set arbitrarily as 10, could be user-defined)
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold} (shape: {filtered_df.shape}):")
        display(filtered_df.head())
        # Normalize this field (z-score)
        mean = filtered_df[numeric_field_id].mean()
        std = filtered_df[numeric_field_id].std()
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - mean) / std
        print(f"\nNormalized {numeric_field_id} (z-score):")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Attempt grouping by a categorical/text field
        group_field_candidates = [c for c in df.columns if c != numeric_field_id]
        group_field = None
        for col in group_field_candidates:
            if pd.api.types.is_string_dtype(df[col]) or pd.api.types.is_categorical_dtype(df[col]):
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
            print(f"\nGrouped mean of {numeric_field_id} by {group_field} (showing top 5):")
            display(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
else:
    print("No DataFrames available for EDA.")

## 5. Visualization

Visualize numeric field distributions or relationships. Example: histogram of the selected numeric field, colored/grouped by a text field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Visualize if prior EDA found suitable fields
if dataframes and 'numeric_field_id' in locals():
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(f"{numeric_field_id}")
    plt.ylabel("Frequency")
    plt.show()

    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.ylabel(f"{numeric_field_id}")
        plt.xlabel(f"{group_field}")
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.show()
else:
    print("No numeric field found for plotting.")

## 6. Conclusion

In this notebook, we:
- Loaded and inspected the FAIR^2 dataset using the `mlcroissant` library and referenced all entities by their `@id`s
- Explored available record sets and their schema structure
- Loaded data into Pandas DataFrames for each record set
- Performed filtering, normalization, and simple group-by aggregations on a numeric field
- Visualized key field distributions with histograms and boxplots

Consult the Croissant metadata and field `@id`s for detailed, schema-aware processing and reproducibility. Adapt this notebook to your research context by extracting, filtering, and visualizing fields that best match your analytical questions.